# AI Agents Fundamentals

## What is an AI Agent?

An **AI agent** is a system that uses an LLM as its reasoning engine to:
1. **Perceive** receive observations from the environment
2. **Reason** decide what action to take (using the LLM)
3. **Act** execute the chosen action (call tools, write files, search)
4. **Observe** receive feedback and update state

Unlike a single LLM call (input → output), agents **loop** until the task is complete.

$$\text{Agent} = \text{LLM} + \text{Tools} + \text{Memory} + \text{Loop}$$

---

## Agent Types

| Type | Description | Example |
|------|-------------|--------|
| **Reactive** | Responds to current input only, no memory | Simple chatbot |
| **Deliberative** | Plans ahead, maintains world model | Research agent |
| **Hybrid** | Combines reactive speed + deliberative planning | Most production agents |

---

## The ReAct Framework

**ReAct** (Yao et al., 2022) interleaves reasoning and acting:

$$\text{Thought}_t \rightarrow \text{Action}_t \rightarrow \text{Observation}_t \rightarrow \text{Thought}_{t+1} \rightarrow \ldots$$

```
Thought: The user wants to know the GDP of France. I should search for it.
Action: search("France GDP 2024")
Observation: France GDP is approximately $3.05 trillion (2023)
Thought: I now have the answer.
Final Answer: France's GDP is approximately $3.05 trillion.
```

---

## Tool Use / Function Calling

Tools are functions the LLM can invoke. Defined as JSON schema:

```json
{
  "name": "search_web",
  "description": "Search the web for current information",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {"type": "string", "description": "The search query"}
    },
    "required": ["query"]
  }
}
```

**Parallel tool calls**: Modern LLMs can call multiple tools simultaneously.

---

## Agent Components

```
┌─────────────────────────────────────────┐
│              AI AGENT                   │
│                                         │
│  ┌──────────┐    ┌────────────────────┐ │
│  │   LLM    │────│  Planning Module   │ │
│  │ Backbone │    └────────────────────┘ │
│  └──────────┘                           │
│       │                                 │
│  ┌────▼───────┐    ┌──────────────────┐ │
│  │   Tools    │    │     Memory       │ │
│  │ - Search   │    │ - Short-term     │ │
│  │ - Code     │    │ - Long-term      │ │
│  │ - APIs     │    │ - Episodic       │ │
│  └────────────┘    └──────────────────┘ │
└─────────────────────────────────────────┘
```

---

## Error Handling in Agents

Agents must handle:
- **Tool failures**: API timeouts, invalid responses
- **Hallucinated tool calls**: LLM inventing non-existent tools
- **Infinite loops**: Agent repeating same action
- **Max iterations**: Hard stop after N steps
- **Parsing errors**: Output not matching expected format

In [1]:
# pip install openai
import os, json
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Define Tools ──────────────────────────────────────────────────────────────
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Python math expression"}
                },
                "required": ["expression"]
            }
        }
    }
]

# ── Tool Implementations ──────────────────────────────────────────────────────
def get_weather(city: str) -> dict:
    # Simulated weather API
    weather_db = {
        "london": {"temp": 15, "condition": "cloudy", "humidity": 78},
        "paris": {"temp": 18, "condition": "sunny", "humidity": 65},
        "tokyo": {"temp": 24, "condition": "humid", "humidity": 85},
    }
    return weather_db.get(city.lower(), {"temp": 20, "condition": "unknown"})

def calculate(expression: str) -> dict:
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}

TOOL_MAP = {"get_weather": get_weather, "calculate": calculate}
print("Tools defined")

Tools defined


In [2]:
# ── Agent Loop from Scratch ───────────────────────────────────────────────────
def run_agent(user_message: str, max_iterations: int = 10) -> str:
    messages = [{"role": "user", "content": user_message}]
    
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")
        
        # LLM decides what to do
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        messages.append(msg)
        
        # Check if done
        if response.choices[0].finish_reason == "stop":
            print(f"Agent finished: {msg.content}")
            return msg.content
        
        # Execute tool calls
        if msg.tool_calls:
            for tool_call in msg.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f"Calling tool: {fn_name}({fn_args})")
                
                # Execute the tool
                result = TOOL_MAP[fn_name](**fn_args)
                print(f"Tool result: {result}")
                
                # Add result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })
    
    return "Max iterations reached"

# Run the agent
result = run_agent("What's the weather in Paris and London? Which is warmer and by how much?")
print(f"\nFinal Answer: {result}")


--- Iteration 1 ---


In [3]:
# ── ReAct Pattern with Explicit Reasoning ─────────────────────────────────────
react_system = """
You are an AI assistant that solves problems using Thought-Action-Observation cycles.

Format:
Thought: [your reasoning about what to do next]
Action: tool_name({"arg": "value"})
Observation: [tool result - provided by system]
... repeat as needed ...
Final Answer: [your complete answer]

Available tools: get_weather(city), calculate(expression)
"""

def react_agent(question: str) -> str:
    messages = [
        {"role": "system", "content": react_system},
        {"role": "user", "content": question}
    ]
    
    for _ in range(5):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            max_tokens=300,
            stop=["Observation:"]
        )
        
        text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": text})
        print(text)
        
        if "Final Answer:" in text:
            return text.split("Final Answer:")[-1].strip()
        
        # Parse and execute action
        if "Action:" in text:
            action_line = [l for l in text.split('\n') if l.startswith('Action:')][0]
            # Simplified parsing
            observation = "Observation: Tool executed successfully\n"
            messages.append({"role": "user", "content": observation})
    
    return "No final answer reached"

react_agent("What is the temperature difference between Paris and London?")

## Additional Learning Resources

### Papers
- [ReAct: Synergizing Reasoning and Acting (Yao et al., 2022)](https://arxiv.org/abs/2210.03629)
- [LLM Powered Autonomous Agents (Lilian Weng, 2023)](https://lilianweng.github.io/posts/2023-06-23-llm-agent/)
- [Toolformer (Schick et al., 2023)](https://arxiv.org/abs/2302.04761)
- [WebGPT (Nakano et al., 2021)](https://arxiv.org/abs/2112.09332)

### Frameworks
- [LangChain Agents Docs](https://python.langchain.com/docs/concepts/agents/)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [Anthropic Tool Use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)

### Blogs
- [The Anatomy of Autonomy (Anthropic)](https://www.anthropic.com/research/building-effective-agents)
- [Agent Protocols (Lilian Weng)](https://lilianweng.github.io/posts/2023-06-23-llm-agent/)